In [ ]:
# ====================================================================================================================
# MT2KD-LITE (Enhancement 1) — CELL 2 of 2: STAGE 2 (5 epochs, fresh for this ROUND) + FINAL EVALUATION + summary vs E0
#   Run AFTER Cell 1 in the SAME runtime (uses its data, settings, ROUND and batch size). Results: <run folder>/stage2_round<ROUND>/
#   After an interruption: run Cell 1 (skips finished work), then this cell -> Stage 2 resumes from its last checkpoint.
# ====================================================================================================================
_missing = [n for n in ("S1", "D", "TA", "DRV1", "DRV2", "train_stage2", "SC", "E", "ROUND", "S2_NAME") if n not in globals()]
if _missing:
    raise SystemExit(f"Cell 1 has not run in this runtime (missing: {_missing}). Run Cell 1 first: it skips finished steps, then run this cell.")
if not S1 or not S1.get("finished") or S1["epoch"] < SC["s1_epochs"]:
    raise SystemExit("Stage 1 of this round is not finished: run Cell 1 again until it prints 'STEP 10 COMPLETED'.")
log(f"\n  CELL 2 (ROUND {ROUND}): Stage 2 for {SC['s2_epochs']} epochs from the best Stage-1 model (after {S1['epoch']} Stage-1 epochs) -> {E['drive_out']}/{S2_NAME}")

if "stage1" in TA: TA["stage1"]["student"].cpu(); TA["stage1"]["student_sd"] = None; TA.pop("stage1"); gc.collect(); torch.cuda.empty_cache()   # not needed any more
STEP(11, NSTEP, f"STAGE 2 (round {ROUND}, fresh from the best Stage-1 model) — fine-tuning with labels: ASR (CTC) + AT (FSD50K, 200 classes) + SV (speaker classification)" + (" + copying the 64M final model" if VAR["ta_stage2"] else "")
                + f" (max {SC['s2_epochs']} epochs)")
_s1best = W("stage1", "best.pt")
if not _s1best.exists() and (DRV1.dir / "best.pt").exists(): shutil.copyfile(DRV1.dir / "best.pt", _s1best)
S2 = train_stage2(D, DEV, AMP, DRV2, _s1best)
if S2 is None or not S2["finished"]: raise SystemExit("Stage 2 not finished in this session: run the cell again to continue")
DONE(11, f"STAGE 2 finished after {S2['epoch']} epochs")

STEP(12, NSTEP, "FINAL EVALUATION of the best Stage-2 model on the OFFICIAL test sets: LibriSpeech test-clean WER, FSD50K eval mAP, VoxCeleb1-O EER")
_ck = torch.load(W(S2_NAME, "best.pt") if W(S2_NAME, "best.pt").exists() else DRV2.dir / "best.pt", map_location="cpu", weights_only=False)
_st, _h1 = build_joint_model(DEV); _st.load_state_dict(_ck["student"]); _hf = build_stage2_heads(_h1, D, DEV); _hf.load_state_dict(_ck["heads"]); _st.eval(); _hf.eval()
FINAL = dict(best_stage2_epoch=_ck["state"]["hist"][-1]["epoch"] if _ck["state"]["hist"] else None)
FINAL["asr_dev_clean"] = asr_wer(_st, _hf, LS_DV, DEV, AMP)
try:
    _tc = MA.inventory("test-clean").reset_index(drop=True)                                 # official test-clean (used ONLY here, once)
    MA.materialize(_tc, "test-clean"); FINAL["asr_test_clean"] = asr_wer(_st, _hf, _tc, DEV, AMP)
except Exception as e: log(f"  (test-clean WER not computed: {type(e).__name__}: {str(e)[:160]})")
FINAL["at_val_mAP_200"] = at_map200(_st, _hf, "val", list(FSD_VA), FSD_VA_Y, DEV, AMP)
if E["fsd_eval_split"] and "eval" in FSD["splits"]:
    _ei = list(range(len(FSD["splits"]["eval"]["fnames"]))); MB.ensure_audio(FSD, "eval", _ei)
    FINAL["at_eval_mAP_200"] = at_map200(_st, _hf, "eval", FSD["splits"]["eval"]["fnames"], FSD["splits"]["eval"]["y"], DEV, AMP)
FINAL["sv_selection_eer"] = sel_eer(_st, _hf["msv"], D, DEV, AMP)
if _have_trials:
    _Ete = MC.student_embeddings(_st, _hf["msv"], TE, None, DEV, AMP); FINAL["sv_official"] = MC.official_eer(_Ete, TE, "MT2KD STUDENT (after Stage 2)")
FINAL["teacher_sv_official"] = TEACHER_SV; FINAL["teacher_at_val_mAP_mapped"] = _tm
_np = lambda m: sum(p.numel() for p in m.parameters())
FINAL["model"] = dict(student="Zipformer-S", variant=VARIANT, encoder_params_M=round(_np(_st) / 1e6, 2), stage2_heads_params_M=round(_np(_hf) / 1e6, 3),
                      deployed_params_M=round((_np(_st) + _np(_hf)) / 1e6, 2), size_fp32_MB=round((_np(_st) + _np(_hf)) * 4 / 2**20, 1),
                      inherited_params_M=round((S1.get("inherited_params") or 0) / 1e6, 2))
try:
    if "stage2" not in TA and (Path(LITE["e0_dir"]) / "stage2" / "best.pt").exists():     # S20_plain: load the 64M model only for the speed comparison
        _ck0 = _e0_ckpt("stage2"); _m0, _msv0 = _build_with(ZIP_M); _m0.load_state_dict(_ck0["student"])
        _c0 = torch.nn.Linear(D_TA, len(VOCAB)); _c0.load_state_dict({k[4:]: v for k, v in _ck0["heads"].items() if k.startswith("ctc.")}); TA["stage2"] = dict(student=_m0, heads=dict(ctc=_c0)); del _ck0
    _bm = {"Zipformer-S (this run)": (_st, _hf["ctc"])}
    if "stage2" in TA: _bm["Zipformer-M 64M (E0)"] = (TA["stage2"]["student"], TA["stage2"]["heads"]["ctc"])
    FINAL["cpu_benchmark"] = cpu_benchmark(_bm, LS_DV, LITE["cpu_benchmark_utts"])
except Exception as e: log(f"  (CPU benchmark not run: {type(e).__name__}: {str(e)[:160]})")
try: E0_FINAL = json.load(open(Path(LITE["e0_dir"]) / "stage2" / "final_report.json"))["final"]
except Exception as e: E0_FINAL = None; log(f"  (E0 final_report.json not read: {type(e).__name__})")
DONE(12, "final evaluation finished")

STEP(13, NSTEP, f"SUMMARY: Zipformer-S ({VARIANT}) vs the 64M baseline E0 + reports on Drive")
_o = (FINAL.get("sv_official") or {}); _o = next(iter(_o.values()), {}) if _o else {}
log(f"  ASR  : dev-clean WER {FINAL['asr_dev_clean']['wer'] * 100:.2f}% (CER {FINAL['asr_dev_clean']['cer'] * 100:.2f}%)"
    + (f" | test-clean WER {FINAL['asr_test_clean']['wer'] * 100:.2f}%" if "asr_test_clean" in FINAL else "") + "  [greedy CTC, characters, no language model]")
log(f"  AT   : FSD50K val mAP {FINAL['at_val_mAP_200'] * 100:.2f}%" + (f" | eval mAP {FINAL['at_eval_mAP_200'] * 100:.2f}%" if "at_eval_mAP_200" in FINAL else "") + "  [200 classes]")
log(f"  SV   : VoxCeleb1-O EER {(_o.get('eer') or float('nan')) * 100:.3f}% ({_o.get('status', 'n/a')}) | non-official selection EER {FINAL['sv_selection_eer'] * 100:.2f}%")
def _g(F_, *ks):
    try:
        x = F_
        for k_ in ks: x = x[k_]
        return x
    except Exception: return None
def _sv(F_):
    o = (F_ or {}).get("sv_official") or {}; o = next(iter(o.values()), {}) if o else {}; return o.get("eer")
_rows = [("ASR test-clean WER (greedy)", _g(E0_FINAL, "asr_test_clean", "wer"), _g(FINAL, "asr_test_clean", "wer"), "%"),
         ("ASR dev-clean WER (greedy)", _g(E0_FINAL, "asr_dev_clean", "wer"), _g(FINAL, "asr_dev_clean", "wer"), "%"),
         ("AT FSD50K val mAP (200 cl.)", _g(E0_FINAL, "at_val_mAP_200"), _g(FINAL, "at_val_mAP_200"), "%"),
         ("SV VoxCeleb1-O EER (official)", _sv(E0_FINAL), _sv(FINAL), "%")]
log(f"\n  {'':32s} {'E0 Zipformer-M (64M)':>22s} {'Zipformer-S (' + VARIANT + ')':>26s}")
for nm, a, b_, u in _rows:
    log(f"  {nm:32s} {('%.2f%%' % (a * 100)) if a is not None else 'n/a':>22s} {('%.2f%%' % (b_ * 100)) if b_ is not None else 'n/a':>26s}")
_bmk = FINAL.get("cpu_benchmark") or {}
log(f"  {'deployed parameters':32s} {'~64.35M':>22s} {str(FINAL['model']['deployed_params_M']) + 'M':>26s}")
for nm_, r_ in _bmk.items(): log(f"  CPU speed, {nm_:38s} RTF {r_['rtf']:.4f} ({r_['seconds']} s for {r_['audio_seconds']} s of audio, 2 threads)")
if len(_bmk) == 2:
    _v = list(_bmk.values()); log(f"  -> Zipformer-S is {_v[1]['rtf'] / _v[0]['rtf']:.2f}x faster on CPU than the 64M model (same utterances, same machine)")
FINAL["comparison_vs_E0"] = dict(rows=[dict(metric=n_, e0=a, this=b_) for n_, a, b_, _ in _rows], e0_dir=LITE["e0_dir"])
log(f"  scale '{SCALE}': pilot-size data -> numbers are far from the paper's (which used LibriSpeech 960 h, AudioSet, VoxCeleb1+2 and a transducer);"
    " they are the reference for comparing teachers / enhancements under identical settings.")
DRV2.write_json("final_report.json", dict(stage="final", round=ROUND, stage1_epochs_done=S1["epoch"], stage1_best_epoch=_b["epoch"], stage2_epochs=SC["s2_epochs"],
                                         batch=dict(max_batch_seconds=E["max_batch_seconds"], max_batch_utts=E["max_batch_utts"]), e2e_version=E2E_VERSION, variant=VARIANT, VAR=VAR, lite=LITE, zip_student=ZIP_STUDENT,
                                         scale=SCALE, data_sig=DATA_SIG, base_sig=BASE_SIG, final=FINAL, stage2_history=S2["hist"],
                                         deviations_from_paper=["student: Zipformer-S (~20M) instead of Zipformer-M (enhancement 1)" + ("; teaching assistant = our 64M E0 model; "
                                                                "Conv-Embed + stacks 1, 2, 6 inherited from E0" if VAR["ta_stage1"] else ""),
                                                                "teachers: WavLM-Large / SSLAM / ReDimNet2-B6 (enhancement) instead of Whisper / BEATs / ECAPA",
                                                                "Stage 2 ASR: CTC with characters + greedy decoding instead of a neural transducer",
                                                                "Stage 2 AT on FSD50K (200 classes) instead of AudioSet; SV on VoxCeleb1 (30,000 training utterances)",
                                                                f"rounds: Stage 1 trained in 3 rounds of {S1_EPOCHS_PER_ROUND} epochs (90 total); each round fine-tunes a fresh {S2_EPOCHS_PER_ROUND}-epoch Stage 2 (paper: 15)",
                                                                f"batch: {E['max_batch_seconds']:.0f} s of audio per batch (measured to fit the GPU; paper ~1000 s)",
                                                                "batches contain one task each (interleaved); every dataset is used completely in every epoch",
                                                                "fixed loss balancing by initial loss values; no checkpoint averaging"]))
DRV2.wait(); DRV1.wait()
DONE(13, f"ROUND {ROUND} DONE. Reports: {E['drive_out']}/stage1/final_report.json and {E['drive_out']}/{S2_NAME}/final_report.json")
if ROUND < 3: log(f"  NEXT ROUND: set ROUND = {ROUND + 1} at the top of Part 2 and run this cell again (Stage 1 continues to epoch {S1_EPOCHS_PER_ROUND * (ROUND + 1)}).")
else: log("  All 3 rounds finished: 90 Stage-1 epochs, as in the base paper.")


